In [1]:
from pathlib import Path
import os


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    return start


def load_env_file() -> None:
    repo_root = _find_repo_root(Path.cwd())
    env_path = repo_root / ".env"
    example_path = repo_root / ".env.example"
    target = env_path if env_path.exists() else example_path
    if not target.exists():
        raise FileNotFoundError(
            f"Expected either {env_path} or {example_path} to exist."
        )

    with target.open() as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key, value)

    print(f"Loaded environment variables from {target.relative_to(repo_root)}")


load_env_file()


Loaded environment variables from .env


In [2]:
from langchain.agents import create_agent

/Users/thory/miniconda3/envs/tiny-ml-pt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def get_weather(city: str) -> str:
    """Get weather for a given city"""
    return f"Its always sunny in {city}"

agent = create_agent(
    model = 'openai:gpt-4o-mini',
    tools = [get_weather],
    prompt = 'You are a helpful assistant',
)

In [4]:
agent.invoke(
    {"message": [{"role":"user", "content": "what is the weather in sf"}]}
)

{'messages': [AIMessage(content='How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 46, 'total_tokens': 54, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CMeOYN6LeXrg84Hdnf1uNpEAhK9uV', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--091aeae4-268a-4146-ba94-6e4851c4e8a2-0', usage_metadata={'input_tokens': 46, 'output_tokens': 8, 'total_tokens': 54, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}

## Weather Forecasting Agent

In [5]:
# Step 1: System Prompt - The Agent's initial instructions or personality.
system_prompt = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean whereever they are, use the get_user_location tool to find their location."""

In [ ]:
# Step 2: Create tools - tools are functions that can be called, they interact with external data to get stuff done.
from langchain_core.tools import tool
import random

def get_weather_for_location(city: str) -> str:
    '''Get weather for a given city'''
    conditions = random.choice(['sunny', 'rainy', 'cloudy'])
    return f'It is {conditions} in {city}'

from langchain_core.runnables import RunnableConfig

# A lookup table for demo purposes
USER_LOCATION = {
    "1":"Florida",
    "2":"SF"
}

'''
@tool decorator turns Python callables into LangChain `StructuredTool` objects
that the agent can discover and invoke. It can then use LangChain's tool metadata 
like names, descriptions, config injections.
'''
@tool
def get_user_location(config: RunnableConfig) -> str:
    '''Retrieve user information'''
    user_id = config.get("configurable", {}).get("user_id")
    return USER_LOCATION[user_id]


In [ ]:
# Step 3: Configure the model
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-4o-mini",
    temperature=0,
)

In [9]:
# Step 4: Define response format
from dataclasses import dataclass

@dataclass
class WeatherResponse:
    conditions: str
    punny_response: str

In [10]:
# Step 5: Add memory for the agent to remember conversation history
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [ ]:
# Step 6: Bring it all together
agent = create_agent(
    model=model,
    prompt=system_prompt,
    tools=[get_user_location, get_weather_for_location],
    response_format=WeatherResponse,
    checkpointer=checkpointer
)

# config = {"configurable": {"thread_id": "1"}}
# context = {"user_id": "1"}

'''
`config` is the run metadata shared across every runnable (models, tools, graphs).
`config` has reserved keys like "configurable", "run_name", "tags", "metadata", "callbacks".
"configurable" is a catch-all for values we want to read back inside the graph or tools.
"thread_id" must be supplied when using `InMemorySaver` or any checkpointer - it decides which conversation thread to load.
'''

config = {"configurable": {"thread_id": "1", "user_id": "2"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
)

response['structured_response']

WeatherResponse(conditions='cloudy', punny_response="Looks like SF is still in a cloudy mood! But don't worry, the sun will break through eventually—it's just taking its sweet time!")

**More control over the model using provider's package**

```python
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-5",
    temperature=0.1,
    max_tokens=1000,
    timeout=30
)
agent = create_agent(model, tools=tools)
```

In [12]:

response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=context
)

response['structured_response']

WeatherResponse(conditions='sunny', punny_response="You're welcome! I'm just trying to brighten your day!")